# Langchain Output Parsers (Legacy)

- Output parsers emerged as an early solution to the challenge of obtaining structured output from LLMs.

- Today, most LLMs support structured output natively. In such cases, using output parsers may be unnecessary, and you should leverage the model's built-in capabilities for structured output. Refer to the documentation of your chosen model for guidance on how to achieve structured output directly.

- Output parsers remain valuable when working with models that do not support structured output natively, or when you require additional processing or validation of the model's output beyond its inherent capabilities.

In [2]:

from dotenv import load_dotenv
from langchain_core.exceptions import OutputParserException

# Below available in langchain-classic package, but not in langchain-core package
# from langchain.output_parsers.datetime import DatetimeOutputParser
# from langchain.output_parsers.boolean import BooleanOutputParser
# from langchain.output_parsers import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_openai import ChatOpenAI

In [3]:
load_dotenv()

True

In [4]:
model = "gpt-5.6-sol"

In [5]:
llm = ChatOpenAI(
    model=model,
    temperature=0.0,
    max_tokens=500,
    timeout=500,
    reasoning_effort="low",
    max_retries=2,
)

In [6]:
response = llm.invoke("hello")

In [7]:
response

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help?', 'annotations': [], 'id': 'msg_00da2dd500cfcf5f006ab1465993d087d2b631b936e85b6141', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_00da2dd500cfcf5f006ab1465602fc87d282daa8d22effcee0', 'created_at': 1790002774.0, 'metadata': {}, 'model': 'gpt-5.6-sol', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol'}, id='resp_00da2dd500cfcf5f006ab1465602fc87d282daa8d22effcee0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 11, 'total_tokens': 18, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [8]:
response.content

[{'type': 'text',
  'text': 'Hello! How can I help?',
  'annotations': [],
  'id': 'msg_00da2dd500cfcf5f006ab1465993d087d2b631b936e85b6141',
  'phase': 'final_answer'}]

In [9]:
parser = StrOutputParser()

In [10]:
parser.invoke(response)

'Hello! How can I help?'

## Pydantic

In [14]:
from typing import Annotated

from pydantic import BaseModel, Field

In [15]:
class Performer(BaseModel):
    """Filmography info about an actor/actress"""

    name: Annotated[str, Field(description="name of an actor/actress")]
    film_names: Annotated[
        list[str], Field(description="list of names of films they starred in")
    ]

In [16]:
llm_with_structure = llm.with_structured_output(Performer)

In [17]:
response = llm_with_structure.invoke(
    "Generate the filmography for Anne Hathaway. Top 3 only"
)

response

Performer(name='Anne Hathaway', film_names=['The Devil Wears Prada', 'Les Misérables', 'The Dark Knight Rises'])

In [18]:
response.model_dump_json(indent=2)

'{\n  "name": "Anne Hathaway",\n  "film_names": [\n    "The Devil Wears Prada",\n    "Les Misérables",\n    "The Dark Knight Rises"\n  ]\n}'

In [19]:
parser = PydanticOutputParser(pydantic_object=Performer)

In [20]:
parser.parse(response.model_dump_json())

Performer(name='Anne Hathaway', film_names=['The Devil Wears Prada', 'Les Misérables', 'The Dark Knight Rises'])

In [21]:
response.model_dump_json()

'{"name":"Anne Hathaway","film_names":["The Devil Wears Prada","Les Misérables","The Dark Knight Rises"]}'

In [22]:
misformatted_result = "{'name':'Anne Hathaway','film_names':['The Devil Wears Prada','Les Misérables','The Dark Knight Rises']}"

In [23]:
try:
    parser.parse(misformatted_result)
except OutputParserException as e:
    print(e)

Invalid json output: {'name':'Anne Hathaway','film_names':['The Devil Wears Prada','Les Misérables','The Dark Knight Rises']}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
